# Elbow-Methode mit dem Wine-Datensatz

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_19/wine_elbow_method.ipynb)

In diesem Notebook lernen wir die **Elbow-Methode** kennen – eine visuelle Hilfe, um eine passende Anzahl Cluster für k-Means zu wählen. Ein eindeutiger Knick ist nicht immer erkennbar.

**Ziel:** Eine sinnvolle Anzahl Cluster vorschlagen, ohne die echten Labels zur Clusterbildung zu verwenden. Anschließend vergleichen wir das Ergebnis mit den bekannten Klassen.

**Das Szenario:** Stellt euch vor, ihr bekommt 178 Weine ohne Etiketten. Welche Gruppen lassen sich anhand ihrer chemischen Eigenschaften erkennen?

Die Referenzklassen stehen für drei Rebsorten, nicht für Produzenten. Quelle: [Wine-Datensatz der UCI](https://archive.ics.uci.edu/dataset/109/wine).

## 1. Bibliotheken importieren und Daten laden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Für schönere Plots
plt.style.use('default')
sns.set_palette("husl")

print("🍷 Wine-Datensatz Elbow-Methode Notebook geladen!")

## 2. Wine-Datensatz verstehen

In [ ]:
# Wine-Datensatz aus sklearn laden
wine = load_wine()

print("📊 Wine-Datensatz Überblick:")
print(f"Anzahl Weine: {wine.data.shape[0]}")
print(f"Anzahl chemische Eigenschaften: {wine.data.shape[1]}")
print(f"Rebsorten-Klassen: {wine.target_names}")

# Echte Verteilung (die wir "vergessen" für k-Means)
unique, counts = np.unique(wine.target, return_counts=True)
print("\n🍇 Verteilung der echten Rebsorten-Klassen:")
for i, (name, count) in enumerate(zip(wine.target_names, counts)):
    print(f"  {name}: {count} Weine")

print("\n🔬 Chemische Eigenschaften:")
for i, feature in enumerate(wine.feature_names[:5]):
    print(f"  {i+1}. {feature}")
print("  ... und 8 weitere")

## 3. Daten vorbereiten (Skalierung!)

**Wichtig:** Der Wine-Datensatz hat sehr unterschiedliche Zahlenbereiche (Alkohol etwa 11–15, Proline 278–1680). Ohne Skalierung beeinflussen Merkmale mit großer Streuung die Abstände viel stärker. Hier verwenden wir StandardScaler, um diesen Effekt zu vermeiden.

In [ ]:
# Skalierungsproblem zeigen
print("⚠️ Skalierungsproblem demonstrieren:")
print(f"Alkohol: {wine.data[:, 0].min():.1f} - {wine.data[:, 0].max():.1f} %")
print(f"Proline (Datensatzwerte): {wine.data[:, 12].min():.0f} - {wine.data[:, 12].max():.0f}")
print("Die Zahlenbereiche unterscheiden sich stark; die Merkmale haben unterschiedliche Einheiten.")

# Alle Features skalieren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(wine.data)

print("\n✅ Nach StandardScaler:")
print(f"Mittelwerte (alle ~0): {X_scaled.mean(axis=0)[:3]}")
print(f"Std-Abweichungen (alle ~1): {X_scaled.std(axis=0)[:3]}")
print("→ Jetzt sind alle Features gleichberechtigt!")

## 4. Elbow-Methode durchführen

Wir probieren verschiedene k-Werte aus (k=1 bis k=10). Die **Inertia** ist die Summe der quadrierten Abstände aller Punkte zu ihrem jeweiligen Clusterzentrum. Kleinere Werte bedeuten, dass die Punkte näher an ihren Zentren liegen.

In [ ]:
# Verschiedene k-Werte testen
k_range = range(1, 11)
inertias = []
silhouette_scores = []

print("🔍 Elbow-Methode läuft...")
for k in k_range:
    # k-Means für aktuelles k
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    
    # Inertia speichern
    inertias.append(kmeans.inertia_)
    
    # Silhouette Score (nur für k >= 2)
    if k >= 2:
        sil_score = silhouette_score(X_scaled, cluster_labels)
        silhouette_scores.append(sil_score)
    else:
        silhouette_scores.append(np.nan)  # Für k=1 ist der Score nicht definiert
    
    print(f"k={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}")

print("✅ Elbow-Methode abgeschlossen!")

## 5. Elbow-Plot visualisieren

Wir suchen einen Knick: Ab dort bringen zusätzliche Cluster weniger Verbesserung. Dieser Knick liefert einen Vorschlag für k, keine garantiert beste Lösung.

In [ ]:
# Elbow-Plot erstellen
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Inertia (Elbow-Methode)
ax1.plot(k_range, inertias, 'bo-', markersize=8, linewidth=2, label='Inertia')
ax1.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (Vorschlag)')
ax1.set_xlabel('Anzahl Cluster (k)')
ax1.set_ylabel('Summe quadrierter Abstände (Inertia)')
ax1.set_title('Elbow-Methode: Wo knickt die Kurve ab?')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Annotationen für wichtige Punkte
ax1.annotate('Starker Abfall', xy=(2, inertias[1]), xytext=(2.5, inertias[1] + 50),
            arrowprops=dict(arrowstyle='->', color='orange'), fontsize=10)
ax1.annotate('Ellenbogen?\nk=3', xy=(3, inertias[2]), xytext=(4, inertias[2] + 30),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)

# Plot 2: Silhouette Score
k_range_sil = range(2, 11)  # Silhouette nur für k >= 2
ax2.plot(k_range_sil, silhouette_scores[1:], 'go-', markersize=8, linewidth=2, label='Silhouette Score')
ax2.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (Vorschlag)')
ax2.set_xlabel('Anzahl Cluster (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score: Höher = bessere Trennung')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Besten Silhouette Score markieren
best_k_sil = k_range_sil[np.argmax(silhouette_scores[1:])]
ax2.annotate(f'Maximum\nk={best_k_sil}', 
            xy=(best_k_sil, max(silhouette_scores[1:])), 
            xytext=(best_k_sil + 1, max(silhouette_scores[1:]) + 0.02),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10)

plt.tight_layout()
plt.show()

print("👁️ Interpretation des Elbow-Plots:")
print("• Links: Suche den 'Ellenbogen' - wo die Kurve stark abflacht")
print("• Rechts: Silhouette Score als zusätzliche Bestätigung")
print("• Wir betrachten k=3 als möglichen Knick; dieser Wert wird nicht automatisch berechnet.")
print(f"• Silhouette Score ist maximal bei k={best_k_sil}")

## 6. Detailanalyse: k=2, k=3, k=4 vergleichen

Schauen wir uns die Kandidaten genauer an. Der **ARI** vergleicht die gefundenen Cluster mit den bekannten Rebsorten-Klassen: 1 bedeutet perfekte Übereinstimmung, etwa 0 eine zufällige Zuordnung. Dieser Vergleich ist eine nachträgliche Kontrolle mit Labels.

In [ ]:
# Detailvergleich für k=2, 3, 4
candidate_ks = [2, 3, 4]
results = {}

print("🔬 Detailanalyse der Kandidaten:")
print(f"{'k':<3} {'Inertia':<10} {'Silhouette':<12} {'ARI':<8} {'Interpretation':<15}")
print("-" * 60)

for k in candidate_ks:
    # k-Means anwenden
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    # Metriken berechnen
    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_scaled, labels)
    ari = adjusted_rand_score(wine.target, labels)  # Vergleich mit echten Labels
    
    # Cluster-Größen
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    results[k] = {
        'inertia': inertia,
        'silhouette': silhouette,
        'ari': ari,
        'cluster_sizes': counts
    }
    
    # Qualitätsbewertung
    if ari > 0.8:
        quality = "Exzellent"
    elif ari > 0.6:
        quality = "Sehr gut"
    elif ari > 0.4:
        quality = "Gut"
    else:
        quality = "Mäßig"
    
    print(f"{k:<3} {inertia:<10.2f} {silhouette:<12.3f} {ari:<8.3f} {quality:<15}")
    print(f"    Cluster-Größen: {counts}")

print("\n💡 Erkenntnisse:")
print("• k=2: Einfache Teilung, aber übersieht Feinstrukturen")
print(f"• k=3: Passt hier gut zu den Rebsorten-Klassen (ARI={results[3]['ari']:.3f}).")
print("• k=4: Ergibt eine feinere Aufteilung, passt hier aber weniger gut zu den Referenzklassen.")

## 7. Finale Entscheidung: k=3 anwenden

Wir wählen für dieses Beispiel k=3: ein plausibler Knick im Elbow-Plot und der höchste Silhouette-Score unter den getesteten Werten. Das ist eine begründete Wahl, keine allgemein optimale Clusterzahl.

In [ ]:
# Finales k-Means mit k=3
print("🏆 Finales k-Means Clustering mit k=3:")
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
final_labels = kmeans_final.fit_predict(X_scaled)
final_centers = kmeans_final.cluster_centers_

# Ergebnisse analysieren
final_ari = adjusted_rand_score(wine.target, final_labels)
final_silhouette = silhouette_score(X_scaled, final_labels)

print("📊 Finale Bewertung:")
print(f"• Adjusted Rand Index: {final_ari:.3f} (1.0 = perfekte Übereinstimmung)")
print(f"• Silhouette Score: {final_silhouette:.3f} (höher = kompaktere, besser getrennte Cluster)")

# Cluster-Größen vs. echte Verteilung
print("\n📈 Cluster-Verteilung:")
unique_final, counts_final = np.unique(final_labels, return_counts=True)
print("k-Means gefunden:")
for cluster, count in zip(unique_final, counts_final):
    print(f"  Cluster {cluster}: {count} Weine")

print("\nEchte Rebsorten-Klassen:")
unique_real, counts_real = np.unique(wine.target, return_counts=True)
for i, (name, count) in enumerate(zip(wine.target_names, counts_real)):
    print(f"  {name}: {count} Weine")

print("\n🎯 Fazit:")
if final_ari > 0.8:
    print("✅ Die gefundenen Cluster stimmen weitgehend mit den Rebsorten-Klassen überein.")
elif final_ari > 0.6:
    print("✅ Sehr gut! k-Means findet die Hauptstrukturen.")
else:
    print("⚠️ Mäßig. Die Daten sind komplexer als erwartet.")

print("Elbow-Plot und Silhouette-Score sprechen in diesem Beispiel für k=3.")

## 8. Bonus: Confusion Matrix - Wer wurde wie zugeordnet?

Schauen wir genauer, welche Weine k-Means welchem Cluster zugeordnet hat. Clusternummern sind beliebige Namen. Zur Kontrolle ordnen wir jedem Cluster seine häufigste Rebsorten-Klasse zu. Nur dieser Vergleich verwendet die echten Labels, nicht das Clustering selbst.

In [ ]:
# Confusion Matrix zwischen k-Means und echten Labels
from sklearn.metrics import confusion_matrix

# Einfache Zuordnung: Für jeden k-Means Cluster das häufigste echte Label finden
print("🔍 k-Means Cluster zu echten Labels zuordnen...")
cluster_to_label = {}

for cluster in range(3):
    # Alle Weine in diesem k-Means Cluster
    cluster_mask = final_labels == cluster
    cluster_true_labels = wine.target[cluster_mask]
    
    # Häufigstes echtes Label in diesem Cluster
    unique_labels, counts = np.unique(cluster_true_labels, return_counts=True)
    most_common_label = unique_labels[np.argmax(counts)]
    
    cluster_to_label[cluster] = most_common_label
    print(f"  k-Means Cluster {cluster} → {wine.target_names[most_common_label]} ({max(counts)}/{len(cluster_true_labels)} Weine)")

# Labels entsprechend zuordnen
final_labels_mapped = np.array([cluster_to_label[label] for label in final_labels])

# Confusion Matrix mit zugeordneten Labels
cm = confusion_matrix(wine.target, final_labels_mapped)

# Visualisierung
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=wine.target_names,
           yticklabels=wine.target_names)
plt.xlabel('Dem Cluster zugeordnete Rebsorten-Klasse')
plt.ylabel('Echte Rebsorten-Klasse')
plt.title('Confusion Matrix: k-Means vs. Echte Labels (zugeordnet)')
plt.show()

print("🧐 Confusion Matrix Interpretation:")
print("• Zeilen: Echte Rebsorten-Klassen")
print("• Spalten: Den Clustern zugeordnete Rebsorten-Klassen") 
print("• Diagonale: Korrekt zugeordnete Weine")
print("• Off-Diagonale: 'Verwechslungen'")

# Anteil richtig zugeordneter Weine pro Referenzklasse
print("\n📊 Zuordnungsqualität pro Rebsorten-Klasse:")
for i, class_name in enumerate(wine.target_names):
    correct = cm[i, i] if i < len(cm) else 0
    total = sum(cm[i, :]) if i < len(cm) else 0
    accuracy = correct / total if total > 0 else 0
    print(f"  {class_name}: {correct}/{total} = {accuracy:.2%} korrekt")

## 9. Zusammenfassung: Was haben wir gelernt?

In [ ]:
print("🎓 Wichtige Erkenntnisse aus der Elbow-Methode:")
print()
print("1️⃣ Elbow-Methode Vorgehen:")
print("   • Verschiedene k-Werte testen (k=1,2,3,...)")
print("   • Inertia plotten (Summe quadrierter Abstände zu den Clusterzentren)")
print("   • 'Ellenbogen' suchen - wo Verbesserung stark abnimmt")
print()
print("2️⃣ Zusätzliche Metriken nutzen:")
print("   • Silhouette Score für Cluster-Qualität")
print("   • Bei bekannten Labels: ARI für Vergleich")
print("   • Cluster-Größen sollten sinnvoll sein")
print()
print("3️⃣ Praktische Tipps:")
print("   • Skalierung prüfen: Hier verwenden wir StandardScaler wegen der verschiedenen Zahlenbereiche.")
print("   • Mehrere random_state testen")
print("   • Domain-Wissen einbeziehen")
print("   • Visualisierung hilft beim Verstehen")
print()
print("4️⃣ Unser Ergebnis:")
print("   • Elbow-Plot und Silhouette-Score → k=3 ist hier eine sinnvolle Wahl.")
print(f"   • ARI = {final_ari:.3f} (Übereinstimmung mit den Referenzklassen)")
print("   • K-Means verwendet keine Labels; sie dienen nur zur nachträglichen Kontrolle.")
print()
print("🚀 Die Elbow-Methode ist ein mächtiges Werkzeug zur")
print("   Auswahl einer passenden Cluster-Anzahl, kein automatischer Beweis!")